# GoPay Google Play Review - TF-IDF

**Author:** Muhammad Razan Parisya Putra  
**Notebook:** `04 - TF-IDF`

This notebook continues from the preprocessing covered in [4-Gopay-Review-Sentiment Analisys.ipynb](https://drive.google.com/file/d/1wtuxEspDw2gAQXJZ_mK-1F1hOTGuxKRM/view?usp=sharing)

#### Installing and importing libraries

In [ ]:
pip install -U nltk

In [ ]:
pip install tensorflow_hub

In [ ]:
pip install xgboost

In [ ]:
import pandas as pd
import numpy as np
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import tensorflow as tf
import tensorflow_hub as hub
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
nltk.download('stopwords')
nltk.download('words')
nltk.download('wordnet')
nltk.download('punkt')

#### Load the GoPay dataset for sentiment analysis

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

In [ ]:
df_gopay_sentiment = pd.read_csv('/content/drive/MyDrive/Tugas 1/Dataset/gopay_reviews_sentiment.csv')
df_gopay_sentiment

In [ ]:
df_gopay_sentiment.dropna(subset=['final_text', 'sentiment'])
df_gopay_sentiment.info()

#### Split Data
x = feature, y = target

In [ ]:
x = df_gopay_sentiment['final_text']
y = df_gopay_sentiment['sentiment']

In [ ]:
label_encode = LabelEncoder()
Y_encoded = label_encode.fit_transform(df_gopay_sentiment['sentiment'])
Y_encoded

In [ ]:
CLASS_NAMES = list(label_encode.classes_)
print(f'Mapping LabelEncoder: {CLASS_NAMES} -> {label_encode.transform(CLASS_NAMES)}')
print(f'Total number of classes: {len(CLASS_NAMES)}')

In [ ]:
# Buat DataFrame sementara untuk filtering yang konsisten
temp_df = pd.DataFrame({
    'content': df_gopay_sentiment['final_text'],
    'sentiment_encoded': Y_encoded
})

In [ ]:
# Filter baris yang kontennya valid
temp_df_filtered = temp_df[
    temp_df['content'].notna() &
    (temp_df['content'].astype(str).str.strip() != '')
]

In [ ]:
x_cleaned = temp_df_filtered['content']
Y_cleaned = temp_df_filtered['sentiment_encoded']

In [ ]:
df_cleaned = pd.DataFrame({
    'final_text': x_cleaned,
    'sentiment_encoded': Y_cleaned
})
df_cleaned['sentiment'] = label_encode.inverse_transform(Y_cleaned.astype(int))
df_cleaned.head(10)

### Split the dataset into training and testing sets

In [ ]:
xtrain_cleaned_split, xtest_cleaned_split, ytrain_cleaned_split, ytest_cleaned_split = train_test_split(
    x_cleaned, Y_cleaned,
    test_size=0.2,
    random_state=42,
    stratify=Y_cleaned
)

print(f'Shape of xtrain_cleaned_split: {xtrain_cleaned_split.shape}')
print(f'Shape of xtest_cleaned_split: {xtest_cleaned_split.shape}')
print(f'Shape of ytrain_cleaned_split: {ytrain_cleaned_split.shape}')
print(f'Shape of ytest_cleaned_split: {ytest_cleaned_split.shape}')

### Note: 0 = negative, 1 = neutral, 2 = positive

### Create a vocabulary

In [ ]:
from collections import Counter

word_counts = Counter(' '.join(str(item) for item in xtrain_cleaned_split).split())

### Display each word in the vocabulary along with its count

In [ ]:
# Tampilkan 50 kata teratas saja agar tidak terlalu panjang
for word, count in word_counts.most_common(50):
    print(f'{word}: {count}')

### Classification with various classifiers
#### 1. Linear SVM
#### 2. Logistic Regression (LR)
#### 3. Naive Bayes
#### 4. XGBoost
#### 5. Random Forest

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

#### Initialize classifiers

In [ ]:
svm_classifier = LinearSVC(max_iter=2000, random_state=42)
logistic_regression = LogisticRegression(max_iter=1000, random_state=42)
nb_classifier = MultinomialNB()
xgboost_classifier = XGBClassifier(
    objective='multi:softmax', eval_metric='mlogloss',
    random_state=42, num_class=len(CLASS_NAMES)
)
random_forest_classifier = RandomForestClassifier(
    n_estimators=100, max_depth=3, max_features='sqrt',
    min_samples_leaf=4, bootstrap=True, n_jobs=-1, random_state=42
)

In [ ]:
def evaluate_model(model, xtest_data, ytest_labels, target_names=CLASS_NAMES):
    y_pred = model.predict(xtest_data)
    accuracy  = accuracy_score(ytest_labels, y_pred)
    precision = precision_score(ytest_labels, y_pred, average='weighted', zero_division=0)
    recall    = recall_score(ytest_labels, y_pred, average='weighted', zero_division=0)
    f1        = f1_score(ytest_labels, y_pred, average='weighted', zero_division=0)
    report    = classification_report(ytest_labels, y_pred, target_names=target_names, zero_division=0)
    cm        = confusion_matrix(ytest_labels, y_pred)
    return y_pred, accuracy, precision, recall, f1, report, cm

In [ ]:
# Define batch size
batch_size = 1000

## 1. TF-IDF
TF-IDF & Classification
1. Linear SVM
2. Logistic Regression
3. Naive Bayes
4. XGBoost classifier
5. Random Forest classifier

#### TF-IDF Vectorization

In [ ]:
print('\n' + '='*80)
print('--- 1. EVALUASI MODEL DENGAN TF-IDF EMBEDDINGS SAJA ---')
print('='*80 + '\n')

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

In [ ]:
tfidf_vectorizer.fit(xtrain_cleaned_split.tolist())

In [ ]:
from scipy.sparse import vstack
from timeit import default_timer as timer

# Embedding TF-IDF untuk data train
start = timer()
tfidf_vectorizer_xtrain_sparse_list = []
xtrain_list = xtrain_cleaned_split.tolist()
for i in range(0, len(xtrain_list), batch_size):
    batch = xtrain_list[i:i + batch_size]
    tfidf_vectorizer_xtrain_sparse_list.append(tfidf_vectorizer.transform(batch))

tfidf_vectorizer_xtrain = vstack(tfidf_vectorizer_xtrain_sparse_list)
print(f'Waktu embedding TF-IDF train: {timer() - start:.4f} seconds')

In [ ]:
# Embedding TF-IDF untuk data test
start = timer()
tfidf_vectorizer_xtest_sparse_list = []
xtest_list = xtest_cleaned_split.tolist()
for i in range(0, len(xtest_list), batch_size):
    batch = xtest_list[i:i + batch_size]
    tfidf_vectorizer_xtest_sparse_list.append(tfidf_vectorizer.transform(batch))

tfidf_vectorizer_xtest = vstack(tfidf_vectorizer_xtest_sparse_list)
print(f'Waktu embedding TF-IDF test: {timer() - start:.4f} seconds')

In [ ]:
print(type(tfidf_vectorizer_xtrain))
print(tfidf_vectorizer_xtrain.shape)
print(type(tfidf_vectorizer_xtest))
print(tfidf_vectorizer_xtest.shape)

In [ ]:
# Re-inisialisasi classifier
svm_classifier = LinearSVC(max_iter=2000, random_state=42)
logistic_regression = LogisticRegression(max_iter=1000, random_state=42)
nb_classifier = MultinomialNB()
xgboost_classifier = XGBClassifier(objective='multi:softmax', eval_metric='mlogloss', random_state=42, num_class=len(CLASS_NAMES))
random_forest_classifier = RandomForestClassifier(n_estimators=100, max_depth=3, max_features='sqrt', min_samples_leaf=4, bootstrap=True, n_jobs=-1, random_state=42)

# Training semua model
start = timer()
svm_tfidf = svm_classifier.fit(tfidf_vectorizer_xtrain, ytrain_cleaned_split)
print(f'Waktu training Linear SVM: {timer() - start:.4f} seconds')

start = timer()
lr_tfidf = logistic_regression.fit(tfidf_vectorizer_xtrain, ytrain_cleaned_split)
print(f'Waktu training Logistic Regression: {timer() - start:.4f} seconds')

start = timer()
nb_tfidf = nb_classifier.fit(tfidf_vectorizer_xtrain, ytrain_cleaned_split)
print(f'Waktu training Naive Bayes: {timer() - start:.4f} seconds')

start = timer()
xgboost_tfidf = xgboost_classifier.fit(tfidf_vectorizer_xtrain, ytrain_cleaned_split)
print(f'Waktu training XGBoost: {timer() - start:.4f} seconds')

start = timer()
rfc_tfidf = random_forest_classifier.fit(tfidf_vectorizer_xtrain, ytrain_cleaned_split)
print(f'Waktu training Random Forest: {timer() - start:.4f} seconds')

In [ ]:
print('\n' + '='*80)
print('--- MAKE PREDICTION ---')
print('='*80 + '\n')

### 1. Linear SVM TF-IDF

In [ ]:
print('\n--- Evaluasi Linear SVM (TF-IDF) ---')
start = timer()
y_pred_svm, accuracy_svm, precision_svm, recall_svm, f1_svm, report_svm, confusion_matrix_svm = \
    evaluate_model(svm_tfidf, tfidf_vectorizer_xtest, ytest_cleaned_split, target_names=CLASS_NAMES)
print(f'Waktu prediksi SVM: {timer() - start:.4f} seconds')

In [ ]:
print('\n=== Label Kelas yang Ada ===')
print(set(ytest_cleaned_split))

In [ ]:
print('\nContoh 15 hasil prediksi SVM:')
for actual, predicted in zip(ytest_cleaned_split[:15], y_pred_svm[:15]):
    print(f'Aktual: {actual}, Prediksi SVM: {predicted}')

In [ ]:
print(f'\n=== Evaluasi SVM ===')
print(f'Akurasi SVM: {accuracy_svm:.4f}')
print(f'Presisi SVM: {precision_svm:.4f}')
print(f'Recall SVM: {recall_svm:.4f}')
print(f'Skor F1 SVM: {f1_svm:.4f}')
print('\n=== Laporan Klasifikasi SVM ===')
print(report_svm)
print('\n=== Matriks Kebingungan SVM ===')
print(confusion_matrix_svm)

### 2. Logistic Regression TF-IDF

In [ ]:
print('\n--- Evaluasi Logistic Regression (TF-IDF) ---')
start = timer()
y_pred_lr, accuracy_lr, precision_lr, recall_lr, f1_lr, report_lr, cm_lr = \
    evaluate_model(lr_tfidf, tfidf_vectorizer_xtest, ytest_cleaned_split, target_names=CLASS_NAMES)
print(f'Waktu prediksi Logistic Regression: {timer() - start:.4f} seconds')

In [ ]:
print(f'\nLabel Mapping: {CLASS_NAMES} -> {label_encode.transform(CLASS_NAMES)}')
for actual, predicted in zip(ytest_cleaned_split[:15], y_pred_lr[:15]):
    print(f'Aktual: {actual}, Prediksi Logistic Regression: {predicted}')

In [ ]:
print(f'\nAkurasi Logistic Regression: {accuracy_lr:.4f}')
print(f'Presisi Logistic Regression: {precision_lr:.4f}')
print(f'Recall Logistic Regression: {recall_lr:.4f}')
print(f'Skor F1 Logistic Regression: {f1_lr:.4f}')
print('Laporan Klasifikasi:\n', report_lr)
print('Matriks Kebingungan:\n', cm_lr)
print('-' * 70)

### 3. Naive Bayes TF-IDF

In [ ]:
print('\n--- Evaluasi Naive Bayes (TF-IDF) ---')
start = timer()
y_pred_nb, accuracy_nb, precision_nb, recall_nb, f1_nb, report_nb, cm_nb = \
    evaluate_model(nb_tfidf, tfidf_vectorizer_xtest, ytest_cleaned_split, target_names=CLASS_NAMES)
print(f'Waktu prediksi Naive Bayes: {timer() - start:.4f} seconds')

In [ ]:
print(f'\nLabel Mapping: {CLASS_NAMES} -> {label_encode.transform(CLASS_NAMES)}')
for actual, predicted in zip(ytest_cleaned_split[:15], y_pred_nb[:15]):
    print(f'Aktual: {actual}, Prediksi Naive Bayes: {predicted}')

In [ ]:
print(f'\nAkurasi Naive Bayes: {accuracy_nb:.4f}')
print(f'Presisi Naive Bayes: {precision_nb:.4f}')
print(f'Recall Naive Bayes: {recall_nb:.4f}')
print(f'Skor F1 Naive Bayes: {f1_nb:.4f}')
print('Laporan Klasifikasi:\n', report_nb)
print('Matriks Kebingungan:\n', cm_nb)
print('-' * 70)

### 4. XGBoost TF-IDF

In [ ]:
print('\n--- Evaluasi XGBoost Classifier (TF-IDF) ---')
start = timer()
y_pred_xgboost, accuracy_xgboost, precision_xgboost, recall_xgboost, f1_xgboost, report_xgboost, cm_xgboost = \
    evaluate_model(xgboost_tfidf, tfidf_vectorizer_xtest, ytest_cleaned_split, target_names=CLASS_NAMES)
print(f'Waktu prediksi XGBoost: {timer() - start:.4f} seconds')

In [ ]:
print(f'\nLabel Mapping: {CLASS_NAMES} -> {label_encode.transform(CLASS_NAMES)}')
for actual, predicted in zip(ytest_cleaned_split[:15], y_pred_xgboost[:15]):
    print(f'Aktual: {actual}, Prediksi XGBoost: {predicted}')

In [ ]:
print(f'\nAkurasi XGBoost: {accuracy_xgboost:.4f}')
print(f'Presisi XGBoost: {precision_xgboost:.4f}')
print(f'Recall XGBoost: {recall_xgboost:.4f}')
print(f'Skor F1 XGBoost: {f1_xgboost:.4f}\n')
print('Laporan Klasifikasi:\n', report_xgboost)
print('Matriks Kebingungan:\n', cm_xgboost)
print('-' * 70)

### 5. Random Forest TF-IDF

In [ ]:
print('\n--- Evaluasi Random Forest Classifier (TF-IDF) ---')
start = timer()
y_pred_rfc, accuracy_rfc, precision_rfc, recall_rfc, f1_rfc, report_rfc, cm_rfc = \
    evaluate_model(rfc_tfidf, tfidf_vectorizer_xtest, ytest_cleaned_split, target_names=CLASS_NAMES)
print(f'Waktu prediksi Random Forest: {timer() - start:.4f} seconds')

In [ ]:
print(f'\nLabel Mapping: {CLASS_NAMES} -> {label_encode.transform(CLASS_NAMES)}')
for actual, predicted in zip(ytest_cleaned_split[:15], y_pred_rfc[:15]):
    print(f'Aktual: {actual}, Prediksi Random Forest: {predicted}')

In [ ]:
print(f'\nAkurasi Random Forest: {accuracy_rfc:.4f}')
print(f'Presisi Random Forest: {precision_rfc:.4f}')
print(f'Recall Random Forest: {recall_rfc:.4f}')
print(f'Skor F1 Random Forest: {f1_rfc:.4f}')
print('Laporan Klasifikasi:\n', report_rfc)
print('Matriks Kebingungan:\n', cm_rfc)
print('-' * 70)

### Perbandingan 5 Model dengan FE TF-IDF

In [ ]:
model_names_tfidf = ['LinearSVM', 'LogisticRegression', 'NaiveBayes', 'XGBoost', 'RandomForest']
model_accuracies_tfidf = [accuracy_svm, accuracy_lr, accuracy_nb, accuracy_xgboost, accuracy_rfc]

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(model_names_tfidf, model_accuracies_tfidf,
        color=['royalblue', 'seagreen', 'goldenrod', 'firebrick', 'mediumpurple'])
plt.xlabel('Model')
plt.ylabel('Akurasi')
plt.title('Perbandingan Akurasi Model dengan TF-IDF')
plt.ylim(0.0, 1.0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

for i, acc in enumerate(model_accuracies_tfidf):
    plt.text(i, acc + 0.02, f'{acc:.2f}', ha='center', color='black', fontsize=12)

plt.tight_layout()
plt.show()

# 2. USE
Universal Sentence Encoder & Classification
1. Linear SVM
2. Logistic Regression
3. XGBoost classifier
4. Random Forest classifier

> **Catatan:** Naive Bayes tidak disertakan karena USE menghasilkan nilai negatif (dense embeddings), sedangkan MultinomialNB hanya menerima nilai non-negatif.

In [ ]:
print('\n' + '='*80)
print('--- 2. UNIVERSAL SENTENCE ENCODER (USE) ---')
print('='*80 + '\n')

In [ ]:
print('Memuat model Universal Sentence Encoder...')
start = timer()
use_embed = hub.load('https://tfhub.dev/google/universal-sentence-encoder/4')
print(f'Waktu load USE: {timer() - start:.4f} seconds')

In [ ]:
xtrain_use = []
xtest_use  = []

In [ ]:
print('Melakukan embedding xtrain dengan USE...')
start = timer()
xtrain_list_for_use = xtrain_cleaned_split.tolist()
for i in range(0, len(xtrain_list_for_use), batch_size):
    batch = xtrain_list_for_use[i:i + batch_size]
    xtrain_use.extend(use_embed(batch).numpy())
xtrain_use = np.array(xtrain_use)
print(f'Waktu embedding xtrain USE: {timer() - start:.4f} seconds')

In [ ]:
print('Melakukan embedding xtest dengan USE...')
start = timer()
xtest_list_for_use = xtest_cleaned_split.tolist()
for i in range(0, len(xtest_list_for_use), batch_size):
    batch = xtest_list_for_use[i:i + batch_size]
    xtest_use.extend(use_embed(batch).numpy())
xtest_use = np.array(xtest_use)
print(f'Waktu embedding xtest USE: {timer() - start:.4f} seconds')

In [ ]:
print(f'\nBentuk USE (train): {xtrain_use.shape}')
print(f'Bentuk USE (test): {xtest_use.shape}\n')

Fit classifier

In [ ]:
print('\n--- Melatih Classifier dengan USE ---')
# Naive Bayes tidak disertakan karena tidak kompatibel dengan embedding USE (nilai negatif)
svm_classifier   = LinearSVC(max_iter=2000, random_state=42)
logistic_regression = LogisticRegression(max_iter=1000, random_state=42)
xgboost_classifier  = XGBClassifier(objective='multi:softmax', eval_metric='mlogloss', random_state=42, num_class=len(CLASS_NAMES))
random_forest_classifier = RandomForestClassifier(n_estimators=100, max_depth=3, max_features='sqrt', min_samples_leaf=4, bootstrap=True, n_jobs=-1, random_state=42)

start = timer()
svm_use = svm_classifier.fit(xtrain_use, ytrain_cleaned_split)
print(f'Waktu training Linear SVM (USE): {timer() - start:.4f} seconds')

start = timer()
lr_use = logistic_regression.fit(xtrain_use, ytrain_cleaned_split)
print(f'Waktu training Logistic Regression (USE): {timer() - start:.4f} seconds')

start = timer()
xgboost_use = xgboost_classifier.fit(xtrain_use, ytrain_cleaned_split)
print(f'Waktu training XGBoost (USE): {timer() - start:.4f} seconds')

start = timer()
rfc_use = random_forest_classifier.fit(xtrain_use, ytrain_cleaned_split)
print(f'Waktu training Random Forest (USE): {timer() - start:.4f} seconds')

### 1. Linear SVM with USE

In [ ]:
print('\n--- Evaluasi Linear SVM (USE) ---')
start = timer()
y_pred_svm_use, accuracy_svm_use, precision_svm_use, recall_svm_use, f1_svm_use, report_svm_use, cm_svm_use = \
    evaluate_model(svm_use, xtest_use, ytest_cleaned_split, target_names=CLASS_NAMES)
print(f'Waktu prediksi Linear SVM (USE): {timer() - start:.4f} seconds')

In [ ]:
print(f'\nLabel Mapping: {CLASS_NAMES} -> {label_encode.transform(CLASS_NAMES)}')
for actual, predicted in zip(ytest_cleaned_split[:15], y_pred_svm_use[:15]):
    print(f'Aktual: {actual}, Prediksi SVM dengan USE: {predicted}')

In [ ]:
print(f'\nAkurasi Linear SVM (USE): {accuracy_svm_use:.4f}')
print(f'Presisi Linear SVM (USE): {precision_svm_use:.4f}')
print(f'Recall Linear SVM (USE): {recall_svm_use:.4f}')
print(f'Skor F1 Linear SVM (USE): {f1_svm_use:.4f}')
print('Laporan Klasifikasi:\n', report_svm_use)
print('Matriks Kebingungan:\n', cm_svm_use)
print('-' * 70)

### 2. Logistic Regression with USE

In [ ]:
print('\n--- Evaluasi Logistic Regression (USE) ---')
start = timer()
y_pred_lr_use, accuracy_lr_use, precision_lr_use, recall_lr_use, f1_lr_use, report_lr_use, cm_lr_use = \
    evaluate_model(lr_use, xtest_use, ytest_cleaned_split, target_names=CLASS_NAMES)
print(f'Waktu prediksi Logistic Regression (USE): {timer() - start:.4f} seconds')

In [ ]:
print(f'\nLabel Mapping: {CLASS_NAMES} -> {label_encode.transform(CLASS_NAMES)}')
for actual, predicted in zip(ytest_cleaned_split[:15], y_pred_lr_use[:15]):
    print(f'Aktual: {actual}, Prediksi Logistic Regression dengan USE: {predicted}')

In [ ]:
print(f'\nAkurasi Logistic Regression (USE): {accuracy_lr_use:.4f}')
print(f'Presisi Logistic Regression (USE): {precision_lr_use:.4f}')
print(f'Recall Logistic Regression (USE): {recall_lr_use:.4f}')
print(f'Skor F1 Logistic Regression (USE): {f1_lr_use:.4f}')
print('Laporan Klasifikasi:\n', report_lr_use)
print('Matriks Kebingungan:\n', cm_lr_use)
print('-' * 70)

### 3. XGBoost with USE

In [ ]:
print('\n--- Evaluasi XGBoost Classifier (USE) ---')
start = timer()
y_pred_xgboost_use, accuracy_xgboost_use, precision_xgboost_use, recall_xgboost_use, f1_xgboost_use, report_xgboost_use, cm_xgboost_use = \
    evaluate_model(xgboost_use, xtest_use, ytest_cleaned_split, target_names=CLASS_NAMES)
print(f'Waktu prediksi XGBoost (USE): {timer() - start:.4f} seconds')

In [ ]:
print(f'\nLabel Mapping: {CLASS_NAMES} -> {label_encode.transform(CLASS_NAMES)}')
for actual, predicted in zip(ytest_cleaned_split[:15], y_pred_xgboost_use[:15]):
    print(f'Aktual: {actual}, Prediksi XGBoost dengan USE: {predicted}')

In [ ]:
print(f'\nAkurasi XGBoost (USE): {accuracy_xgboost_use:.4f}')
print(f'Presisi XGBoost (USE): {precision_xgboost_use:.4f}')
print(f'Recall XGBoost (USE): {recall_xgboost_use:.4f}')
print(f'Skor F1 XGBoost (USE): {f1_xgboost_use:.4f}')
print('Laporan Klasifikasi:\n', report_xgboost_use)
print('Matriks Kebingungan:\n', cm_xgboost_use)
print('-' * 70)

### 4. Random Forest with USE

In [ ]:
print('\n--- Evaluasi Random Forest Classifier (USE) ---')
start = timer()
y_pred_rfc_use, accuracy_rfc_use, precision_rfc_use, recall_rfc_use, f1_rfc_use, report_rfc_use, cm_rfc_use = \
    evaluate_model(rfc_use, xtest_use, ytest_cleaned_split, target_names=CLASS_NAMES)
print(f'Waktu prediksi Random Forest (USE): {timer() - start:.4f} seconds')

In [ ]:
print(f'\nLabel Mapping: {CLASS_NAMES} -> {label_encode.transform(CLASS_NAMES)}')
for actual, predicted in zip(ytest_cleaned_split[:15], y_pred_rfc_use[:15]):
    print(f'Aktual: {actual}, Prediksi Random Forest dengan USE: {predicted}')

In [ ]:
print(f'\nAkurasi Random Forest (USE): {accuracy_rfc_use:.4f}')
print(f'Presisi Random Forest (USE): {precision_rfc_use:.4f}')
print(f'Recall Random Forest (USE): {recall_rfc_use:.4f}')
print(f'Skor F1 Random Forest (USE): {f1_rfc_use:.4f}')
print('Laporan Klasifikasi:\n', report_rfc_use)
print('Matriks Kebingungan:\n', cm_rfc_use)
print('-' * 70)

#### Display all four models accuracy (USE)

In [ ]:
model_names_use = ['LinearSVM', 'LogisticRegression', 'XGBoost', 'RandomForest']
model_accuracies_use = [accuracy_svm_use, accuracy_lr_use, accuracy_xgboost_use, accuracy_rfc_use]

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(model_names_use, model_accuracies_use,
        color=['royalblue', 'seagreen', 'firebrick', 'mediumpurple'])
plt.xlabel('Model')
plt.ylabel('Akurasi')
plt.title('Perbandingan Akurasi Model dengan USE')
plt.ylim(0.0, 1.0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

for i, acc in enumerate(model_accuracies_use):
    plt.text(i, acc + 0.02, f'{acc:.2f}', ha='center', color='black', fontsize=12)

plt.tight_layout()
plt.show()

# 3. TF-IDF + Universal Sentence Encoder (USE)
1. Linear SVM
2. Logistic Regression
3. XGBoost classifier
4. Random Forest classifier

#### Combine TF-IDF and Universal Sentence Encoder embeddings

In [ ]:
print('--- Menggabungkan TF-IDF dan USE Embeddings ---')

# Convert sparse TF-IDF ke dense array sebelum digabungkan dengan USE
xtrain_tfidf_dense = tfidf_vectorizer_xtrain.toarray() if hasattr(tfidf_vectorizer_xtrain, 'toarray') else tfidf_vectorizer_xtrain
xtest_tfidf_dense  = tfidf_vectorizer_xtest.toarray()  if hasattr(tfidf_vectorizer_xtest,  'toarray') else tfidf_vectorizer_xtest

In [ ]:
xtrain_use_np = np.array(xtrain_use)
xtest_use_np  = np.array(xtest_use)

In [ ]:
# Gabungkan secara horizontal (concatenate fitur)
xtrain_tfidf_use_combined = np.hstack([xtrain_tfidf_dense, xtrain_use_np])
xtest_tfidf_use_combined  = np.hstack([xtest_tfidf_dense,  xtest_use_np])

In [ ]:
print(f'Bentuk TF-IDF + USE (train): {xtrain_tfidf_use_combined.shape}')
print(f'Bentuk TF-IDF + USE (test) : {xtest_tfidf_use_combined.shape}\n')

#### Fit classifiers

In [ ]:
print('\n--- Melatih Classifier dengan TF-IDF + USE ---')
svm_classifier   = LinearSVC(max_iter=2000, random_state=42)
logistic_regression = LogisticRegression(max_iter=1000, random_state=42)
xgboost_classifier  = XGBClassifier(objective='multi:softmax', eval_metric='mlogloss', random_state=42, num_class=len(CLASS_NAMES))
random_forest_classifier = RandomForestClassifier(n_estimators=100, max_depth=3, max_features='sqrt', min_samples_leaf=4, bootstrap=True, n_jobs=-1, random_state=42)

start = timer()
svm_tfidf_use = svm_classifier.fit(xtrain_tfidf_use_combined, ytrain_cleaned_split)
print(f'Waktu training Linear SVM (TF-IDF + USE): {timer() - start:.4f} seconds')

start = timer()
lr_tfidf_use = logistic_regression.fit(xtrain_tfidf_use_combined, ytrain_cleaned_split)
print(f'Waktu training Logistic Regression (TF-IDF + USE): {timer() - start:.4f} seconds')

start = timer()
xgboost_tfidf_use = xgboost_classifier.fit(xtrain_tfidf_use_combined, ytrain_cleaned_split)
print(f'Waktu training XGBoost (TF-IDF + USE): {timer() - start:.4f} seconds')

start = timer()
rfc_tfidf_use = random_forest_classifier.fit(xtrain_tfidf_use_combined, ytrain_cleaned_split)
print(f'Waktu training Random Forest (TF-IDF + USE): {timer() - start:.4f} seconds')

#### Make predictions

### Linear SVM

In [ ]:
print('\n--- Evaluasi Linear SVM (TF-IDF + USE) ---')
start = timer()
y_pred_svm_tfidf_use, accuracy_svm_tfidf_use, precision_svm_tfidf_use, recall_svm_tfidf_use, f1_svm_tfidf_use, report_svm_tfidf_use, cm_svm_tfidf_use = \
    evaluate_model(svm_tfidf_use, xtest_tfidf_use_combined, ytest_cleaned_split, target_names=CLASS_NAMES)
print(f'Waktu prediksi Linear SVM (TF-IDF + USE): {timer() - start:.4f} seconds')

In [ ]:
print(f'\nLabel Mapping: {CLASS_NAMES} -> {label_encode.transform(CLASS_NAMES)}')
for actual, predicted in zip(ytest_cleaned_split[:15], y_pred_svm_tfidf_use[:15]):
    print(f'Aktual: {actual}, Prediksi SVM dengan TF-IDF + USE: {predicted}')

In [ ]:
print(f'\nAkurasi Linear SVM (TF-IDF + USE): {accuracy_svm_tfidf_use:.4f}')
print(f'Presisi Linear SVM (TF-IDF + USE): {precision_svm_tfidf_use:.4f}')
print(f'Recall Linear SVM (TF-IDF + USE): {recall_svm_tfidf_use:.4f}')
print(f'Skor F1 Linear SVM (TF-IDF + USE): {f1_svm_tfidf_use:.4f}')
print('Laporan Klasifikasi:\n', report_svm_tfidf_use)
print('Matriks Kebingungan:\n', cm_svm_tfidf_use)
print('-' * 70)

### Logistic Regression

In [ ]:
print('\n--- Evaluasi Logistic Regression (TF-IDF + USE) ---')
start = timer()
y_pred_lr_tfidf_use, accuracy_lr_tfidf_use, precision_lr_tfidf_use, recall_lr_tfidf_use, f1_lr_tfidf_use, report_lr_tfidf_use, cm_lr_tfidf_use = \
    evaluate_model(lr_tfidf_use, xtest_tfidf_use_combined, ytest_cleaned_split, target_names=CLASS_NAMES)
print(f'Waktu prediksi Logistic Regression (TF-IDF + USE): {timer() - start:.4f} seconds')

In [ ]:
print(f'\nLabel Mapping: {CLASS_NAMES} -> {label_encode.transform(CLASS_NAMES)}')
for actual, predicted in zip(ytest_cleaned_split[:15], y_pred_lr_tfidf_use[:15]):
    print(f'Aktual: {actual}, Prediksi LR dengan TF-IDF + USE: {predicted}')

In [ ]:
print(f'\nAkurasi Logistic Regression (TF-IDF + USE): {accuracy_lr_tfidf_use:.4f}')
print(f'Presisi Logistic Regression (TF-IDF + USE): {precision_lr_tfidf_use:.4f}')
print(f'Recall Logistic Regression (TF-IDF + USE): {recall_lr_tfidf_use:.4f}')
print(f'Skor F1 Logistic Regression (TF-IDF + USE): {f1_lr_tfidf_use:.4f}')
print('Laporan Klasifikasi:\n', report_lr_tfidf_use)
print('Matriks Kebingungan:\n', cm_lr_tfidf_use)
print('-' * 70)

### XGBoost Classifier

In [ ]:
print('\n--- Evaluasi XGBoost Classifier (TF-IDF + USE) ---')
start = timer()
y_pred_xgb_tfidf_use, accuracy_xgb_tfidf_use, precision_xgb_tfidf_use, recall_xgb_tfidf_use, f1_xgb_tfidf_use, report_xgb_tfidf_use, cm_xgb_tfidf_use = \
    evaluate_model(xgboost_tfidf_use, xtest_tfidf_use_combined, ytest_cleaned_split, target_names=CLASS_NAMES)
print(f'Waktu prediksi XGBoost (TF-IDF + USE): {timer() - start:.4f} seconds')

In [ ]:
print(f'\nLabel Mapping: {CLASS_NAMES} -> {label_encode.transform(CLASS_NAMES)}')
for actual, predicted in zip(ytest_cleaned_split[:15], y_pred_xgb_tfidf_use[:15]):
    print(f'Aktual: {actual}, Prediksi XGBoost dengan TF-IDF + USE: {predicted}')

In [ ]:
print(f'\nAkurasi XGBoost (TF-IDF + USE): {accuracy_xgb_tfidf_use:.4f}')
print(f'Presisi XGBoost (TF-IDF + USE): {precision_xgb_tfidf_use:.4f}')
print(f'Recall XGBoost (TF-IDF + USE): {recall_xgb_tfidf_use:.4f}')
print(f'Skor F1 XGBoost (TF-IDF + USE): {f1_xgb_tfidf_use:.4f}')
print('Laporan Klasifikasi:\n', report_xgb_tfidf_use)
print('Matriks Kebingungan:\n', cm_xgb_tfidf_use)
print('-' * 70)

### Random Forest Classifier

In [ ]:
print('\n--- Evaluasi Random Forest Classifier (TF-IDF + USE) ---')
start = timer()
y_pred_rfc_tfidf_use, accuracy_rfc_tfidf_use, precision_rfc_tfidf_use, recall_rfc_tfidf_use, f1_rfc_tfidf_use, report_rfc_tfidf_use, cm_rfc_tfidf_use = \
    evaluate_model(rfc_tfidf_use, xtest_tfidf_use_combined, ytest_cleaned_split, target_names=CLASS_NAMES)
print(f'Waktu prediksi Random Forest (TF-IDF + USE): {timer() - start:.4f} seconds')

In [ ]:
print(f'\nLabel Mapping: {CLASS_NAMES} -> {label_encode.transform(CLASS_NAMES)}')
for actual, predicted in zip(ytest_cleaned_split[:15], y_pred_rfc_tfidf_use[:15]):
    print(f'Aktual: {actual}, Prediksi Random Forest dengan TF-IDF + USE: {predicted}')

In [ ]:
print(f'\nAkurasi Random Forest (TF-IDF + USE): {accuracy_rfc_tfidf_use:.4f}')
print(f'Presisi Random Forest (TF-IDF + USE): {precision_rfc_tfidf_use:.4f}')
print(f'Recall Random Forest (TF-IDF + USE): {recall_rfc_tfidf_use:.4f}')
print(f'Skor F1 Random Forest (TF-IDF + USE): {f1_rfc_tfidf_use:.4f}')
print('Laporan Klasifikasi:\n', report_rfc_tfidf_use)
print('Matriks Kebingungan:\n', cm_rfc_tfidf_use)
print('-' * 70)

# Visualization
#### Visualisasi Perbandingan Akurasi Model dengan Berbagai Jenis Embedding

In [ ]:
# Classifier yang dibandingkan (tidak termasuk Naive Bayes untuk USE dan TF-IDF+USE)
classifiers = ['Linear SVM', 'Logistic Regression', 'XGBoost', 'Random Forest']

In [ ]:
tfidf_accuracies    = [accuracy_svm,          accuracy_lr,          accuracy_xgboost,     accuracy_rfc]
use_accuracies      = [accuracy_svm_use,       accuracy_lr_use,      accuracy_xgboost_use, accuracy_rfc_use]
tfidf_use_accuracies= [accuracy_svm_tfidf_use, accuracy_lr_tfidf_use,accuracy_xgb_tfidf_use, accuracy_rfc_tfidf_use]

In [ ]:
x = np.arange(len(classifiers))
bar_width = 0.25

plt.figure(figsize=(12, 7))
plt.bar(x - bar_width, tfidf_accuracies,     bar_width, label='TF-IDF',     color='orange')
plt.bar(x,             use_accuracies,        bar_width, label='USE',         color='purple')
plt.bar(x + bar_width, tfidf_use_accuracies,  bar_width, label='TF-IDF+USE',  color='red')

plt.xlabel('Classifier')
plt.ylabel('Akurasi')
plt.title('Perbandingan Akurasi Classifier dengan Berbagai Jenis Embedding')
plt.xticks(x, classifiers)
plt.ylim(0.0, 1.0)
plt.legend()

for i in range(len(classifiers)):
    plt.text(x[i] - bar_width, tfidf_accuracies[i]    + 0.02, f'{tfidf_accuracies[i]:.2f}',     ha='center', fontsize=10)
    plt.text(x[i],             use_accuracies[i]       + 0.02, f'{use_accuracies[i]:.2f}',       ha='center', fontsize=10)
    plt.text(x[i] + bar_width, tfidf_use_accuracies[i] + 0.02, f'{tfidf_use_accuracies[i]:.2f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

#### Kesimpulan:

Dari visualisasi di atas, dapat disimpulkan bahwa dibandingkan dengan embedding TF-IDF saja, embedding USE serta kombinasi TF-IDF+USE menunjukkan kinerja yang lebih baik dalam analisis sentimen pada dataset ulasan GoPay dengan berbagai classifier. Ini mengindikasikan bahwa embedding yang lebih kaya konteks (seperti USE) atau kombinasi embedding berbasis frekuensi dan konteks dapat menghasilkan model klasifikasi sentimen yang lebih akurat.

---
**Reference:**
- https://github.com/FarhanaTeli/Sentiment_Analysis_IMDB
- https://github.com/Wittline/tf-idf
- https://github.com/divaardeliaa/ScrapReviewReliveApp